In [101]:
#pip install torch torchvision torchaudio
#conda install pytorch torchvision -c pytorch

In [102]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

## Components

In [103]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value
        
        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)
        
        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output
    

In [104]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

In [105]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [106]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [107]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

## Model

In [108]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        # Get device from input tensors
        device = src.device
        
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        
        # Create nopeak_mask on the SAME DEVICE as tgt (fix device mismatch)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length, device=device), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

## Training

In [109]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

def extract_translation(batch):
    return {
        "src_text": batch["translation"]["en"],
        "tgt_text": batch["translation"]["de"],
    }

def length_filter(example, max_length=100):
    return (len(example["src_text"].split()) < max_length and
            len(example["tgt_text"].split()) < max_length)

def get_translation_loaders(batch_size=32, max_length=100, lang="de-en", dataset_name="wmt14", max_train_samples=50000, max_val_samples=5000):
    """
    Load translation dataset (WMT14 or OPUS100). IWSLT2017 no longer supported.
    
    Args:
        batch_size: Batch size for DataLoader
        max_length: Max sequence length
        lang: Language pair (e.g., "de-en")
        dataset_name: "wmt14", "wmt16", "wmt17", "opus100"
        max_train_samples: Limit training samples (e.g., 50000 for single GPU)
        max_val_samples: Limit validation samples (e.g., 5000)
    """
    # Load dataset and limit size for single GPU
    dataset = load_dataset(dataset_name, lang)
    print(f"Original dataset size: train={len(dataset['train'])}, val={len(dataset['validation'])}")
    
    # Subset for GPU memory
    dataset["train"] = dataset["train"].select(range(min(max_train_samples, len(dataset["train"]))))
    dataset["validation"] = dataset["validation"].select(range(min(max_val_samples, len(dataset["validation"]))))
    print(f"Reduced dataset size: train={len(dataset['train'])}, val={len(dataset['validation'])}")
    
    dataset = dataset.map(extract_translation, remove_columns=["translation"])
    dataset["train"] = dataset["train"].filter(lambda ex: length_filter(ex, max_length))

    # Tokenizer (for en->de)
    tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")

    def tokenize_batch(batch):
        inputs = tokenizer(batch["src_text"], truncation=True, padding="max_length", max_length=max_length)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(batch["tgt_text"], truncation=True, padding="max_length", max_length=max_length)
        inputs["labels"] = labels["input_ids"]
        return inputs

    tokenized_train = dataset["train"].map(tokenize_batch, batched=True, remove_columns=dataset["train"].column_names)
    tokenized_val = dataset["validation"].map(tokenize_batch, batched=True, remove_columns=dataset["validation"].column_names)

    tokenized_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    tokenized_val.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    def collate_fn(batch):
        input_ids = torch.stack([item["input_ids"] for item in batch])
        attention_mask = torch.stack([item["attention_mask"] for item in batch])
        labels = torch.stack([item["labels"] for item in batch])
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

    # pin_memory=True helps transfer data to GPU faster
    train_loader = DataLoader(tokenized_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=0)
    val_loader = DataLoader(tokenized_val, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=0)
    return train_loader, val_loader, tokenizer

In [110]:
# ===== OPTIMIERT FÜR SINGLE GPU =====
# Hyperparameters (reduced for 1 GPU)
max_seq_length = 64  # Reduced from 100 to save memory
batch_size = 8       # Reduced from 32; adjust based on GPU VRAM
d_model = 256        # Reduced from 512 (kann auch 512 sein, probieren!)
num_heads = 4        # Reduced from 8 (muss d_model % num_heads == 0)
num_layers = 3       # Reduced from 6 (schneller training)
d_ff = 512           # Reduced from 2048
dropout = 0.1

# Dataset: 50k training samples (vs ~4M in full WMT14)
train_loader, val_loader, tokenizer = get_translation_loaders(
    batch_size=batch_size, 
    max_length=max_seq_length, 
    dataset_name="wmt14",
    max_train_samples=50000,  # Limit to 50k; reduce to 10000 if OOM
    max_val_samples=5000      # 5k validation
)

# Vocabulary sizes from tokenizer
src_vocab_size = tokenizer.vocab_size
tgt_vocab_size = tokenizer.vocab_size
print(f"Vocab size: {src_vocab_size}")

# Build model
transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
transformer.to(device)

# Count parameters
total_params = sum(p.numel() for p in transformer.parameters())
print(f"Model parameters: {total_params:,}")

# Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
optimizer = optim.Adam(transformer.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

# Convert tokenizer IDs to GPU tensors (fix device mismatch)
pad_token_id = torch.tensor(tokenizer.pad_token_id, device=device)
bos_token_id = torch.tensor(
    tokenizer.bos_token_id if tokenizer.bos_token_id is not None 
    else (tokenizer.cls_token_id if tokenizer.cls_token_id is not None 
    else tokenizer.pad_token_id), 
    device=device
)
eos_token_id = torch.tensor(
    tokenizer.eos_token_id if tokenizer.eos_token_id is not None 
    else tokenizer.pad_token_id, 
    device=device
)

def shift_right(labels, bos_token_id, device):
    """Shift labels right and add BOS token (ensure GPU placement)"""
    dec_input = labels.clone().to(device)
    dec_input = torch.roll(dec_input, shifts=1, dims=1)
    dec_input[:, 0] = bos_token_id.item()  # Use .item() to get scalar value
    return dec_input

best_eval_loss = float('inf')

# Training loop
print(f"\n{'='*60}")
print(f"Training on {device} with {len(train_loader)} batches per epoch")
print(f"{'='*60}\n")

transformer.train()
for epoch in range(10):
    running_loss = 0.0
    for batch_idx, batch in enumerate(train_loader):
        src = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        decoder_input = shift_right(labels, bos_token_id, device)

        optimizer.zero_grad()
        output = transformer(src, decoder_input)
        logits = output.contiguous().view(-1, tgt_vocab_size)
        target = labels.contiguous().view(-1)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Print every 50 batches
        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch {batch_idx + 1}/{len(train_loader)} | Loss: {loss.item():.4f}")
    
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch:{epoch+1} | Train Loss: {avg_loss:.4f}")

    # Validation
    transformer.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            src = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            decoder_input = shift_right(labels, bos_token_id, device)
            output = transformer(src, decoder_input)
            logits = output.contiguous().view(-1, tgt_vocab_size)
            target = labels.contiguous().view(-1)
            loss = criterion(logits, target)
            val_loss += loss.item()
    val_avg = val_loss / len(val_loader)
    print(f"Validation Loss: {val_avg:.4f}")
    if val_avg < best_eval_loss:
        best_eval_loss = val_avg
        best_model = transformer.state_dict()
        print(f"  ✓ Best model saved (loss: {val_avg:.4f})")
    transformer.train()

Original dataset size: train=4508785, val=3000
Reduced dataset size: train=50000, val=3000
Vocab size: 58101
Device: cuda
Model parameters: 48,633,333

Training on cuda with 6079 batches per epoch

  Batch 50/6079 | Loss: 9.1776
  Batch 100/6079 | Loss: 8.0390
  Batch 150/6079 | Loss: 7.6835
  Batch 200/6079 | Loss: 7.2636
  Batch 250/6079 | Loss: 6.8258
  Batch 300/6079 | Loss: 6.9586
  Batch 350/6079 | Loss: 6.9478
  Batch 400/6079 | Loss: 6.7737
  Batch 450/6079 | Loss: 6.9880
  Batch 500/6079 | Loss: 6.7598
  Batch 550/6079 | Loss: 6.4379
  Batch 600/6079 | Loss: 6.3233
  Batch 650/6079 | Loss: 6.2258
  Batch 700/6079 | Loss: 6.3298
  Batch 750/6079 | Loss: 6.5558
  Batch 800/6079 | Loss: 6.5888
  Batch 850/6079 | Loss: 6.3239
  Batch 900/6079 | Loss: 6.3278
  Batch 950/6079 | Loss: 6.2182
  Batch 1000/6079 | Loss: 6.1983
  Batch 1050/6079 | Loss: 6.1736
  Batch 1100/6079 | Loss: 6.3970
  Batch 1150/6079 | Loss: 6.2002
  Batch 1200/6079 | Loss: 6.0036
  Batch 1250/6079 | Loss: 6.41

## Evaluation

In [111]:
# Evaluation & Inference
transformer.eval()

# Beispiel: Texte übersetzen
example_texts = [
    "Hello, how are you?",
    "I love machine learning.",
    "The weather is nice today."
]

print("="*60)
print("Translation Examples")
print("="*60)

with torch.no_grad():
    for text in example_texts:
        # Encode source text (ENSURE GPU)
        src_tokens = tokenizer(text, return_tensors="pt", padding="max_length", max_length=max_seq_length, truncation=True)
        src_ids = src_tokens["input_ids"].to(device)
        
        # Initialize target with BOS token (ENSURE GPU)
        tgt_ids = torch.full((1, max_seq_length), tokenizer.pad_token_id, dtype=torch.long, device=device)
        tgt_ids[0, 0] = bos_token_id.item()
        
        # Greedy decoding (für Demo; beam search wäre besser)
        for i in range(1, max_seq_length):
            # Forward pass
            logits = transformer(src_ids, tgt_ids)
            # Get next token (greedy: max probability)
            next_token = torch.argmax(logits[0, i-1, :], dim=-1)
            tgt_ids[0, i] = next_token.item()
            # Stop if EOS reached
            if next_token.item() == tokenizer.eos_token_id or next_token.item() == tokenizer.pad_token_id:
                break
        
        # Decode to text
        translation = tokenizer.decode(tgt_ids[0].cpu(), skip_special_tokens=True)
        print(f"EN: {text}")
        print(f"DE: {translation}\n")

# Final validation loss
print("\n" + "="*60)
print("Final Validation Loss")
print("="*60)
transformer.eval()
final_val_loss = 0.0
with torch.no_grad():
    for batch in val_loader:
        src = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        decoder_input = shift_right(labels, bos_token_id, device)
        output = transformer(src, decoder_input)
        logits = output.contiguous().view(-1, tgt_vocab_size)
        target = labels.contiguous().view(-1)
        loss = criterion(logits, target)
        final_val_loss += loss.item()

final_val_avg = final_val_loss / len(val_loader)
print(f"Final Validation Loss: {final_val_avg:.4f}")
print(f"Best Validation Loss: {best_eval_loss:.4f}")

# Model info
print("\n" + "="*60)
print("Model Summary")
print("="*60)
print(f"Device: {device}")
print(f"Total Parameters: {total_params:,}")
print(f"Model Dtype: {list(transformer.parameters())[0].dtype}")
print(f"Model Device: {list(transformer.parameters())[0].device}")

Translation Examples
EN: Hello, how are you?
DE: Wie sollen Sie also?

EN: I love machine learning.
DE: Ich habe mich auf die Möglichkeit.

EN: The weather is nice today.
DE: Die heute ist heute abend.


Final Validation Loss
Final Validation Loss: 6.2876
Best Validation Loss: 6.2876

Model Summary
Device: cuda
Total Parameters: 48,633,333
Model Dtype: torch.float32
Model Device: cuda:0


In [ ]:
transformer.eval()

while True:
    print("give me a sentence to translate or STOP:\n")
    input_text = input()
    if input_text.strip().upper() == "STOP":
        break
    else:
        print("="*60)
        print("Translation Examples")
        print("="*60)

        with torch.no_grad():
    
            # Encode source text (ENSURE GPU)
            src_tokens = tokenizer(input_text, return_tensors="pt", padding="max_length", max_length=max_seq_length, truncation=True)
            src_ids = src_tokens["input_ids"].to(device)
            
            # Initialize target with BOS token (ENSURE GPU)
            tgt_ids = torch.full((1, max_seq_length), tokenizer.pad_token_id, dtype=torch.long, device=device)
            tgt_ids[0, 0] = bos_token_id.item()
            
            # Greedy decoding (für Demo; beam search wäre besser)
            for i in range(1, max_seq_length):
                # Forward pass
                logits = transformer(src_ids, tgt_ids)
                # Get next token (greedy: max probability)
                next_token = torch.argmax(logits[0, i-1, :], dim=-1)
                tgt_ids[0, i] = next_token.item()
                # Stop if EOS reached
                if next_token.item() == tokenizer.eos_token_id or next_token.item() == tokenizer.pad_token_id:
                    break
            
            # Decode to text
            translation = tokenizer.decode(tgt_ids[0].cpu(), skip_special_tokens=True)
            print(f"EN: {input_text}")
            print(f"DE: {translation}\n")
